# 16 тонких вращающихся лучей: Fourier-teacher и открытая Wasserstein + tube дистилляция

По умолчанию выбирается новый **generic FourierFMMLP teacher**: 100 000 updates на 65 536
точках из **16 вращающихся лучей**, hidden 256 × 4, time embedding 32. Он использует общие
Cartesian/polar Fourier features, без аналитического ответа или зашитого числа лучей.
`TEACHER_PRESET="auto"` выбирает Fourier YAML, когда он есть в checkout; иначе доступен
plain-MLP C16 baseline. `TEACHER_PRESET="plain"` явно выбирает этот отдельный comparison.
Старый teacher на четырёх лучах не используется ни в одном preset.

**Plain C16 teacher после 100 000 updates не выучил тонкие моды:** validation mean absolute
phase residual ≈0.198 сектора при N32 и ≈0.193 при N128, real ≈0.0194. Малые общие MMD/SWD
этого не исключают. Это диагностический baseline. Само наличие 100 000 updates у нового
Fourier-parent тоже не подтверждает качество: перед калибровкой ниже открыта geometry QC.

Четыре режима: **M0 = MSE**, **M1 = MSE + tube**, **E0 = MSE + Sinkhorn endpoint**,
**E1 = MSE + Sinkhorn endpoint + tube**. Ученик стартует из точных frozen EMA-весов родителя.
Учитель делает два Euler-микрошага, ученик один: **32 → 16 NFE**; `t=1` — шум, `t=0` — данные.
Кондиционер и CFG здесь не используются. Default — **exploratory: 200 updates × 4 метода,
seed 101**. Зарегистрированная серия — **2 500 updates × 4 метода × 3 paired seeds**.

Teacher target, tube-retargeting, признаки, loss, `backward`, optimizer и EMA видны в коде.
Sinkhorn остаётся проверенной численной функцией. До калибровки проверяются native class,
архитектура и Fourier settings выбранного preset, manifest, completed updates и hash весов.
Если matching parent отсутствует, выполнение останавливается; готовый plain parent не
подменяет отсутствующий Fourier parent. Веса и manifest скачиваются отдельными файлами.
В ноутбуке нет wheel, embedded weights или сохранённых outputs. Checkpoints, PNG/SVG и GIF
сохраняются в отдельном exploratory-каталоге.

In [ ]:
import os
import subprocess
import sys
from importlib.metadata import version

from packaging.version import Version

GIT_BRANCH = "research/wasserstein-tube-replication"
MIN_VERSION = "0.4.2"
GIT_SOURCE = (
    "diffusion-lab[replication,interactive] @ "
    "git+https://github.com/Vovanm88/ddpm_vibe_research.git@" + GIT_BRANCH
)
if os.environ.get("DIFFUSION_LAB_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", GIT_SOURCE])
if Version(version("diffusion-lab")) < Version(MIN_VERSION):
    raise RuntimeError("This notebook needs diffusion-lab >= 0.4.2; install the branch and restart the kernel")


In [ ]:
import copy
import hashlib
import json
import math
import tempfile
import time
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Image, clear_output, display

from diffusion_lab.visualization import plot_rainbow_trajectories, save_figure, save_trajectory_gif
from recheck.config import load_config
from recheck.data import make_toy_dataset
from recheck.evaluate import fit_feature_reference, fixed_projection_bank, rays_quality
from recheck.fm import learning_rate, load_model, sample_euler
from recheck.losses import production_sinkhorn, sliced_wasserstein_loss
from recheck.rays_analysis import analyse_rays, plot_rays_diagnostics
from recheck.runtime import (
    RNGStreams,
    atomic_json,
    choose_device,
    model_hash,
    save_checkpoint,
    tensor_hash,
)


In [ ]:
DEVICE = "cpu"
THREADS = 4
SEED = 101
DATA_SEED = 41
N_CLASSES = 16
N_TRAIN = 65536
N_VALIDATION = 8192
UPDATES = 200                       # exploratory; registered CLI study uses 2500 × 3 seeds
BATCH_SIZE = 128                     # also OT support size; no hidden accumulation
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP = 20
EMA_DECAY = 0.99
GRAD_CLIP = 1.0
TEACHER_STEPS = 32
STUDENT_STEPS = 16
TUBE_RHO = 0.10
DISTRIBUTION_FAMILY = "sinkhorn"      # "sinkhorn" or "swd"
REPRESENTATION = "endpoint"          # "endpoint", "displacement", "joint"
LAMBDA_W = 1.0
GAMMA = 1.0
EPSILON_ALPHA = 0.05
CALIBRATION_SAMPLES = 256
SWD_PROJECTIONS = 128
CHECKPOINT_EVERY = 50
RESUME = False                      # True restores exactly this run/config
EVAL_SAMPLES = 1024
EVAL_SEED = 730002                   # separate fixed validation noise, independent of NFE
GIF_PARTICLES = 32
MAKE_GIFS = True
SHOW_WIDGETS = True
CALIBRATION_SEED = 410016          # fixed train-only stream, independent of student seed
REFERENCE_SEED = 730003            # fixed train-only MMD/SWD reference
TEACHER_PRESET = "auto"            # "auto", "fourier", or explicit underfit comparison "plain"
CONFIG_PATH = Path("replication/configs/many_rays_fourier_cpu.yaml")
PLAIN_CONFIG_PATH = Path("replication/configs/many_rays_cpu.yaml")
STUDY_ID = "rays16_fourier_teacher_v1"  # setup resolves this from the selected config
PUBLISHED_TEACHER_PATH = Path("replication/demo") / STUDY_ID / "teacher/ema_model"
OUTPUT_DIR = Path("outputs/many_rays_wasserstein_tube_seed101_exploratory")


In [ ]:
def resolve_teacher_profile(repo_dir, preset, fourier_config, plain_config):
    """auto chooses the newer config if it exists; parent availability never changes preset."""
    paths = {"fourier": Path(fourier_config), "plain": Path(plain_config)}
    if preset == "auto":
        preset = "fourier" if (Path(repo_dir) / paths["fourier"]).is_file() else "plain"
    if preset not in paths:
        raise ValueError("TEACHER_PRESET must be auto, fourier, or plain")
    config_path = paths[preset]
    return preset, config_path, load_config(Path(repo_dir) / config_path)


def select_completed_teacher(candidates, registered):
    """Select a matching completed native parent; never substitute another toy teacher."""
    rejected = []
    for candidate in candidates:
        candidate = Path(candidate)
        try:
            manifest = json.loads((candidate.parent / "manifest.json").read_text())
            architecture = json.loads((candidate / "config.json").read_text())
            recorded = manifest["config"]
            if manifest.get("status") != "completed":
                raise ValueError("teacher is not completed")
            if manifest.get("updates_completed", 0) < max(100000, registered["teacher"]["updates"]):
                raise ValueError("teacher has fewer than 100000 completed updates")
            if recorded.get("protocol_id") != registered["protocol_id"]:
                raise ValueError("wrong teacher study")
            for key in ("name", "n_classes", "n_train", "seed"):
                if recorded["data"][key] != registered["data"][key]:
                    raise ValueError(f"wrong teacher data.{key}")
            if recorded["data"]["name"] != "rotating_rays" or recorded["data"]["n_classes"] != 16:
                raise ValueError("this notebook requires a 16-ray parent")
            if recorded["teacher"] != registered["teacher"]:
                raise ValueError("teacher training profile differs from the registered long run")
            for key in ("hidden_dim", "num_layers", "time_embedding_dim"):
                expected = registered["model"][key]
                if recorded["model"][key] != expected or architecture[key] != expected:
                    raise ValueError(f"wrong teacher model.{key}")
            kind = registered["model"]["kind"]
            expected_class = {"mlp": "FMMLP", "fourier_mlp": "FourierFMMLP"}.get(kind)
            if (expected_class is None or recorded["model"]["kind"] != kind
                    or architecture["_class_name"] != expected_class or architecture["input_dim"] != 2):
                raise ValueError("wrong native model class for the selected unconditional 2-D profile")
            if kind == "fourier_mlp":
                for config_key, native_key in (("spatial_frequencies", "spatial_frequencies"),
                        ("polar_features", "polar_features"), ("spatial_epsilon", "epsilon")):
                    expected = registered["model"][config_key]
                    if recorded["model"][config_key] != expected or architecture[native_key] != expected:
                        raise ValueError(f"wrong teacher Fourier setting {config_key}")
            if not manifest.get("teacher_sha256") or not manifest.get("data_sha256"):
                raise ValueError("teacher provenance hashes are missing")
            return candidate, manifest
        except (OSError, ValueError, KeyError, TypeError) as error:
            rejected.append(f"{candidate}: {error}")
    raise FileNotFoundError("Finish/publish the matching completed 100000-update 16-ray teacher first. "
                            "Calibration and student training have not started.\n" + "\n".join(rejected))


def load_verified_teacher(candidates, registered, device):
    path, manifest = select_completed_teacher(candidates, registered)
    model = load_model(path, device).eval().requires_grad_(False)
    if model_hash(model) != manifest["teacher_sha256"]:
        raise ValueError("Native teacher weights do not match teacher_sha256 in the manifest")
    return model, path, manifest


In [ ]:
# Find the checkout, or download source + native weights separately from the notebook.
REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
repo_dir = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "src" / "recheck").is_dir()), None)
if repo_dir is None:
    repo_dir = Path(tempfile.mkdtemp(prefix="many-rays-source-")) / "repository"
    subprocess.check_call(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH,
                           REPOSITORY_URL, str(repo_dir)])
selected_preset, CONFIG_PATH, registered_config = resolve_teacher_profile(
    repo_dir, TEACHER_PRESET, CONFIG_PATH, PLAIN_CONFIG_PATH)
STUDY_ID = registered_config["protocol_id"]
PUBLISHED_TEACHER_PATH = Path("replication/demo") / STUDY_ID / "teacher/ema_model"
if (registered_config["data"]["name"] != "rotating_rays"
        or registered_config["data"]["n_classes"] != N_CLASSES
        or N_CLASSES != 16):
    raise ValueError("This notebook requires the registered 16-ray long-teacher profile")
if (N_TRAIN, DATA_SEED) != (registered_config["data"]["n_train"], registered_config["data"]["seed"]):
    raise ValueError("Keep the teacher train data fixed; name a separate study to change it")
print("Selected teacher preset:", selected_preset, "config:", CONFIG_PATH)
if selected_preset == "plain":
    print("Plain C16 is an underfit comparison. Inspect phase QC before exploratory training.")
teacher_candidates = [
    repo_dir / registered_config["output_dir"] / "teacher/ema_model",  # completed live run first
    repo_dir / PUBLISHED_TEACHER_PATH,
]
torch.set_num_threads(THREADS)
device = choose_device(DEVICE)
teacher, teacher_path, teacher_manifest = load_verified_teacher(teacher_candidates, registered_config, device)
teacher_hash = model_hash(teacher)
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
train = make_toy_dataset("rotating_rays", N_TRAIN, DATA_SEED, split="train", n_classes=N_CLASSES)
validation = make_toy_dataset("rotating_rays", N_VALIDATION, DATA_SEED,
                              split="validation", n_classes=N_CLASSES)
if tensor_hash(train) != teacher_manifest["data_sha256"]:
    raise ValueError("Regenerated train data does not match the completed teacher manifest")
assert TEACHER_STEPS % STUDENT_STEPS == 0 and STUDENT_STEPS > 0
MICROSTEPS = TEACHER_STEPS // STUDENT_STEPS
H = 1.0 / STUDENT_STEPS
if DISTRIBUTION_FAMILY not in {"sinkhorn", "swd"}:
    raise ValueError("choose sinkhorn or swd")
if REPRESENTATION not in {"endpoint", "displacement", "joint"}:
    raise ValueError("unknown representation")
print("Native teacher class:", type(teacher).__name__)
print("Teacher:", teacher_path, "completed updates:", teacher_manifest["updates_completed"])
print("Frozen teacher hash:", teacher_hash, "unconditional; labels never enter a model")
print("Exploratory student updates:", UPDATES, "seed:", SEED, "outputs:", OUTPUT_DIR)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(validation[:, 0], validation[:, 1], s=2, alpha=.4)
ax.set(title="Fixed real validation · 16 unconditional rotating rays", aspect="equal")
plt.show()


## Проверка длинного родителя до калибровки

При 16 лучах сектор узкий: поворот одного луча около 20°, а не длинная спираль.
Вычитаем известный зависящий от радиуса поворот и сравниваем **phase residual в долях
сектора**, ближайший луч и radial CDF с независимым real validation. У центра угол
неустойчив, поэтому phase/occupancy используют `r > .25R`; radial и coverage — все точки.
Real reference тоже содержит шум: нулевая толщина и строгое `r≤R` не являются целью.
Эта диагностика показывает заполнение промежутков и пропавшие моды, дополняя MMD/SWD.

In [ ]:
geometry_teacher_noise = torch.randn(EVAL_SAMPLES, 2,
    generator=torch.Generator().manual_seed(EVAL_SEED))
geometry_teacher_samples = sample_euler(teacher, geometry_teacher_noise, steps=TEACHER_STEPS)["samples"]
teacher_geometry = analyse_rays(geometry_teacher_samples, validation, n_rays=N_CLASSES)
atomic_json(OUTPUT_DIR / "teacher_geometry_validation.json", {
    "scope": "independent_validation_geometry_before_student_training",
    "teacher_sha256": teacher_hash, "teacher_nfe": TEACHER_STEPS, "diagnostics": teacher_geometry})
fig, _ = plot_rays_diagnostics(geometry_teacher_samples, validation, n_rays=N_CLASSES,
                              title=f"Completed C16 teacher · N{TEACHER_STEPS}")
save_figure(fig, OUTPUT_DIR / "teacher_geometry_validation")
plt.show()
print("Parent geometry relative to noisy real validation:", teacher_geometry["metrics"])


## Один переход: две скорости учителя → одна скорость ученика

`a_used` — один общий вход для обеих моделей. В tube-режимах `a_used = a + rho*H*eta`,
где `eta ~ N(0,I)` **на каждую координату**, без нормирования длины. Учитель каждый раз
заново интегрируется из `a_used`. Конец локального перехода не обязательно является
концом всей генерации. Ученик получает только `(x,t)`: длина `H` находится снаружи модели.

In [ ]:
@torch.no_grad()
def teacher_microsteps(model, anchor, t, width, microsteps=2):
    state = anchor.detach()
    h = width / microsteps
    for k in range(microsteps):
        velocity = model(state, float(t) - k * h).sample
        state = state - h * velocity                 # t decreases toward the data
    return state.detach()


def transition_features(anchor, endpoint, representation="endpoint", gamma=1.0):
    dimension = endpoint[0].numel()
    a = anchor.flatten(1)
    y = endpoint.flatten(1)
    if representation == "endpoint":
        return y / math.sqrt(dimension)
    displacement = y - a
    if representation == "displacement":
        return displacement / math.sqrt(dimension)
    if representation == "joint":
        return torch.cat((math.sqrt(gamma) * a, displacement), dim=1) / math.sqrt(dimension)
    raise ValueError("unknown representation")


def distribution_penalty(phi_student, phi_teacher, family, epsilon, streams):
    if family == "sinkhorn":
        # Canonical squared-cost, KL regularization, both student self-term gradients.
        # Raises if either marginal/solver fails; no clamp or nan_to_num shortcut.
        return production_sinkhorn(phi_student.float(), phi_teacher.float(), epsilon)
    if family == "swd":
        # New random directions use only their own stream, preserving paired data/t/noise.
        return sliced_wasserstein_loss(phi_student.float(), phi_teacher.float(),
                                      generator=streams["swd"], num_directions=SWD_PROJECTIONS)
    raise ValueError("unknown distribution family")


## Калибровка до обучения: только frozen teacher и чистый train

Для каждого из 16 интервалов и каждого representation вычисляем median off-diagonal
teacher-to-teacher squared cost, затем `epsilon = .05 * max(median, 1e-8)`.
Эта таблица одинакова для tube/no-tube, не зависит от student и не обновляется во время training.
Это train-only калибровка, а не подбор на validation или test.

In [ ]:
calibration_streams = RNGStreams(CALIBRATION_SEED)
calibration = {representation: [] for representation in ("endpoint", "displacement", "joint")}
calibration_trace = []
with torch.no_grad():
    for j in range(STUDENT_STEPS):
        t = 1.0 - j * H
        indices = torch.randint(len(train), (CALIBRATION_SAMPLES,), generator=calibration_streams["data"])
        x0 = train[indices].to(device)
        x1 = torch.randn(x0.shape, generator=calibration_streams["noise"]).to(device)
        anchor = (1 - t) * x0 + t * x1
        endpoint = teacher_microsteps(teacher, anchor, t, H, MICROSTEPS)
        calibration_trace.append(tensor_hash(anchor))
        for representation in calibration:
            phi_teacher = transition_features(anchor, endpoint, representation, GAMMA).detach().cpu().double()
            costs = torch.cdist(phi_teacher, phi_teacher).square()
            upper = torch.triu_indices(len(costs), len(costs), offset=1, device=costs.device)
            median_cost = float(torch.quantile(costs[upper[0], upper[1]], .5))
            if median_cost < 1e-8:
                print("WARNING: degenerate teacher cost", representation, j, median_cost)
            calibration[representation].append({"interval": j, "t": t, "H": H,
                "median_cost": median_cost, "epsilon": EPSILON_ALPHA * max(median_cost, 1e-8)})
atomic_json(OUTPUT_DIR / "calibration.json", {"representations": calibration,
    "clean_train_anchor_hashes": calibration_trace, "alpha": EPSILON_ALPHA,
    "teacher_sha256": teacher_hash, "support_size": CALIBRATION_SAMPLES, "seed": CALIBRATION_SEED, "n_classes": N_CLASSES})
print("Endpoint epsilon:", [round(row["epsilon"], 6) for row in calibration["endpoint"]])


## Открытый training loop: student init → target → loss → backward → optimizer → EMA

Все методы стартуют **точной копией одних и тех же teacher-весов**. Для каждого метода создаём
одинаковые независимые RNG streams: data, interval, noise, tube, SWD. Даже `rho=0` потребляет
тот же `eta`. Дополнительная SWD-случайность не меняет ни данные, ни времена, ни шумы.
`pairing_trace` проверяет идентичность последовательностей после прогона.

Главный checkpoint — финальная EMA после одинакового числа updates. Loss не делится на `H²`.
Линейные anchors ниже — интерполянты независимых пар data/noise; это не индивидуальные пути
истинного rollout учителя. Меняя источник anchors, нужно повторять всю сравниваемую четвёрку.

In [ ]:
# Alternate representation/family becomes explicitly named exploratory runs.
dist_label = "E" if (DISTRIBUTION_FAMILY, REPRESENTATION) == ("sinkhorn", "endpoint") else (
    DISTRIBUTION_FAMILY + "_" + REPRESENTATION + "_"
)
method_specs = [("M0", None, 0.0), ("M1", None, TUBE_RHO),
                (dist_label + "0", DISTRIBUTION_FAMILY, 0.0),
                (dist_label + "1", DISTRIBUTION_FAMILY, TUBE_RHO)]
trained, training_logs, pairing_traces, run_manifests = {}, {}, {}, {}

for method_name, family, rho in method_specs:
    student = copy.deepcopy(teacher).train().requires_grad_(True)
    ema = copy.deepcopy(teacher).eval().requires_grad_(False)
    assert model_hash(student) == teacher_hash == model_hash(ema)
    optimizer = torch.optim.AdamW(student.parameters(), lr=LEARNING_RATE,
                                  betas=(.9, .999), weight_decay=WEIGHT_DECAY)
    streams = RNGStreams(SEED)                     # RESET identically for every method
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = method_dir / "training.pt"
    contract = {"study_id": STUDY_ID, "scope": "exploratory_visible_loop", "n_classes": N_CLASSES,
                "teacher_sha256": teacher_hash, "train_sha256": tensor_hash(train),
                "seed": SEED, "updates": UPDATES, "batch_size": BATCH_SIZE,
                "teacher_steps": TEACHER_STEPS, "student_steps": STUDENT_STEPS,
                "family": family, "representation": REPRESENTATION, "rho": rho,
                "lambda_w": LAMBDA_W, "gamma": GAMMA, "lr": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY, "warmup": WARMUP, "ema_decay": EMA_DECAY,
                "grad_clip": GRAD_CLIP, "swd_projections": SWD_PROJECTIONS,
                "calibration": calibration[REPRESENTATION], "anchor_source": "interpolant"}
    history, pairing_trace, completed = [], [], 0
    if checkpoint_path.exists():
        if not RESUME:
            raise FileExistsError(f"{checkpoint_path} exists: use RESUME=True or a new OUTPUT_DIR")
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        if saved["contract"] != contract:
            raise ValueError("Resume contract differs: use a new output directory")
        student.load_state_dict(saved["student"])
        ema.load_state_dict(saved["ema"])
        optimizer.load_state_dict(saved["optimizer"])
        for state in optimizer.state.values():
            for key, value in state.items():
                if isinstance(value, torch.Tensor):
                    state[key] = value.to(device)
        streams.load_state_dict(saved["streams"])
        torch.set_rng_state(saved["torch_rng"])
        if device.type == "cuda" and saved["cuda_rng"]:
            torch.cuda.set_rng_state_all(saved["cuda_rng"])
        if device.type == "mps" and saved.get("mps_rng") is not None:
            torch.mps.set_rng_state(saved["mps_rng"])
        history, pairing_trace = saved["history"], saved["pairing_trace"]
        completed = saved["updates_completed"]
    elif RESUME:
        print(method_name, "has no checkpoint: start its paired run from the teacher")

    def commit_checkpoint(status):
        save_checkpoint(checkpoint_path, {"contract": contract,
            "student": student.state_dict(), "ema": ema.state_dict(),
            "optimizer": optimizer.state_dict(), "streams": streams.state_dict(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else [],
            "mps_rng": torch.mps.get_rng_state() if device.type == "mps" else None,
            "history": history, "pairing_trace": pairing_trace,
            "updates_completed": completed, "status": status})

    started = time.perf_counter()
    previous_seconds = history[-1]["training_seconds"] if history else 0.
    commit_checkpoint("running")
    try:
        for update in range(completed, UPDATES):
            j = int(torch.randint(STUDENT_STEPS, (), generator=streams["interval"]))
            t = 1.0 - j * H                        # ONE shared time for the whole OT support
            indices = torch.randint(len(train), (BATCH_SIZE,), generator=streams["data"])
            x0 = train[indices].to(device)
            x1 = torch.randn(x0.shape, generator=streams["noise"]).to(device)
            eta = torch.randn(x0.shape, generator=streams["tube"]).to(device)
            anchor = (1 - t) * x0 + t * x1
            a_used = anchor + rho * H * eta
            # RECOMPUTE the target FROM THIS perturbed input, never reuse a clean target.
            teacher_endpoint = teacher_microsteps(teacher, a_used, t, H, MICROSTEPS)
            student_endpoint = a_used - H * student(a_used, t).sample
            pair = (student_endpoint - teacher_endpoint.detach()).square().mean()
            dist = pair.new_zeros(())
            if family is not None and LAMBDA_W != 0:
                phi_student = transition_features(a_used, student_endpoint, REPRESENTATION, GAMMA)
                phi_teacher = transition_features(a_used, teacher_endpoint, REPRESENTATION, GAMMA).detach()
                dist = distribution_penalty(phi_student, phi_teacher, family,
                                             calibration[REPRESENTATION][j]["epsilon"], streams)
            loss = pair + LAMBDA_W * dist
            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite loss: stop this series and inspect the cause")
            lr = learning_rate(update, UPDATES, WARMUP, LEARNING_RATE)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP,
                                                       error_if_nonfinite=True)
            optimizer.step()
            with torch.no_grad():
                for ema_parameter, parameter in zip(ema.parameters(), student.parameters()):
                    ema_parameter.lerp_(parameter.detach(), 1 - EMA_DECAY)
                for ema_buffer, buffer in zip(ema.buffers(), student.buffers()):
                    ema_buffer.copy_(buffer)
            completed = update + 1
            pairing_trace.append(hashlib.sha256((str(j) + tensor_hash(indices) +
                                  tensor_hash(x1) + tensor_hash(eta)).encode()).hexdigest())
            history.append({"update": completed, "pair_loss": float(pair.detach()),
                "dist_loss": float(dist.detach()), "total_loss": float(loss.detach()),
                "grad_norm": float(grad_norm), "lr": lr, "interval": j, "t": t,
                "training_seconds": previous_seconds + time.perf_counter() - started})
            if completed == 1 or completed % CHECKPOINT_EVERY == 0:
                print(method_name, completed, "pair=", round(float(pair.detach()), 7),
                      "dist=", round(float(dist.detach()), 7))
                commit_checkpoint("running")
    except BaseException as error:
        # Keep the LAST FULLY COMMITTED weights/RNG; discard any partial update.
        saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        saved["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
        save_checkpoint(checkpoint_path, saved)
        atomic_json(method_dir / "manifest.json", contract | {
            "method": method_name, "status": saved["status"],
            "updates_completed": saved["updates_completed"], "error": str(error),
            "recovery": "RESUME=True restores last fully committed update"})
        raise
    commit_checkpoint("completed")
    assert model_hash(teacher) == teacher_hash and all(p.grad is None for p in teacher.parameters())
    ema.save_pretrained(method_dir / "ema_model", safe_serialization=True)
    manifest = contract | {"method": method_name, "status": "completed",
        "updates_completed": completed, "ema_sha256": model_hash(ema),
        "rng_seeds": streams.seeds, "teacher_calls_per_update": MICROSTEPS,
        "teacher_calls_training": completed * MICROSTEPS,
        "micro_batch": BATCH_SIZE, "optimizer_effective_batch": BATCH_SIZE,
        "ot_support_size": BATCH_SIZE, "ot_computed": family is not None and LAMBDA_W != 0,
        "precision": "float32 network; float64 converged Sinkhorn",
        "training_seconds": history[-1]["training_seconds"]}
    atomic_json(method_dir / "manifest.json", manifest)
    atomic_json(method_dir / "history.json", history)
    trained[method_name] = ema
    training_logs[method_name] = history
    pairing_traces[method_name] = pairing_trace
    run_manifests[method_name] = manifest

assert all(trace == pairing_traces["M0"] for trace in pairing_traces.values())
print("Paired data/t/noise/tube traces match for all four methods; teacher unchanged.")


## Quality и fidelity — отдельные измерения

Все методы и обе frozen baseline получают **один и тот же initial noise**; NFE не участвует
в seed. Quality измеряем относительно независимых real validation: MMD² с train-only
RBF reference и фиксированный SWD. Отрицательную unbiased MMD² не обрезаем.
Fidelity — paired endpoint MSE относительно original teacher на 32 шагах.
Сходство с учителем не означает улучшение относительно данных.

In [ ]:
initial_noise = torch.randn(EVAL_SAMPLES, 2, generator=torch.Generator().manual_seed(EVAL_SEED))
reference = fit_feature_reference(train, seed=REFERENCE_SEED, bandwidth_samples=2000,
                                  standardize=False)
projection_bank = fixed_projection_bank(2, 1024, seed=REFERENCE_SEED + 1)
teacher_label = f"Original N{TEACHER_STEPS}"
models_to_compare = {teacher_label: (teacher, TEACHER_STEPS),
                     f"Original N{STUDENT_STEPS}": (teacher, STUDENT_STEPS)}
models_to_compare.update({f"{name} N{STUDENT_STEPS}": (model, STUDENT_STEPS) for name, model in trained.items()})
generations, metrics = {}, []
for label, (model, steps) in models_to_compare.items():
    generated = sample_euler(model, initial_noise, steps=steps, record_history=True)
    generations[label] = generated
    quality = rays_quality(generated["samples"], validation, reference, projection_bank,
                           block_size=256)
    fidelity = float((generated["samples"] - generations[teacher_label]["samples"]).square().mean())
    row = {"model": label, "nfe": generated["nfe"], "initial_noise_sha256": generated["initial_noise_sha256"],
           "paired_mse_to_original_teacher": fidelity, **quality}
    metrics.append(row)
    print(label, "SWD=", round(row["sliced_wasserstein"], 6),
          "MMD2(real)=", round(row["mmd2_real"], 7), "paired MSE=", round(fidelity, 7))
assert len({row["initial_noise_sha256"] for row in metrics}) == 1
atomic_json(OUTPUT_DIR / "metrics.json", {"scope": "exploratory_16_rotating_rays_validation_one_paired_seed",
    "study_id": STUDY_ID, "n_classes": N_CLASSES, "teacher_sha256": teacher_hash,
    "seed": SEED, "reference": reference.to_dict(), "rows": metrics})
np.savez_compressed(OUTPUT_DIR / "samples_and_histories.npz", initial_noise=initial_noise.numpy(),
                    **{label.replace(" ", "_"): result["history"].numpy()
                       for label, result in generations.items()})


In [ ]:
geometry_reports = {
    label: analyse_rays(result["samples"], validation, n_rays=N_CLASSES)
    for label, result in generations.items()
}
atomic_json(OUTPUT_DIR / "geometry_comparison_validation.json", {
    "scope": "exploratory_common_noise_independent_validation", "n_classes": N_CLASSES,
    "teacher_sha256": teacher_hash, "rows": geometry_reports})
for label, result in geometry_reports.items():
    print(label, result["metrics"])


def render_geometry(label):
    fig, _ = plot_rays_diagnostics(generations[label]["samples"], validation,
                                  n_rays=N_CLASSES, title=label + " · fixed real reference")
    plt.show()


if SHOW_WIDGETS:
    geometry_selector = widgets.Dropdown(options=list(generations), description="Geometry")
    display(widgets.VBox([geometry_selector,
        widgets.interactive_output(render_geometry, {"label": geometry_selector})]))
else:
    render_geometry(teacher_label)
    render_geometry(list(generations)[-1])


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
plots = [("Real validation", validation)] + [(name, result["samples"]) for name, result in generations.items()]
limits = np.concatenate([points.numpy() for _, points in plots])
low, high = limits.min(0) - .08, limits.max(0) + .08
for ax, (label, points) in zip(axes.flat, plots):
    ax.scatter(points[:, 0], points[:, 1], s=2, alpha=.5)
    ax.set(title=label, xlim=(low[0], high[0]), ylim=(low[1], high[1]), aspect="equal")
for ax in axes.flat[len(plots):]:
    ax.axis("off")
save_figure(fig, OUTPUT_DIR / "generation_comparison")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for name, history in training_logs.items():
    axes[0].plot([r["update"] for r in history], [r["pair_loss"] for r in history], label=name)
    axes[1].plot([r["update"] for r in history], [r["dist_loss"] for r in history], label=name)
axes[0].set(title="Raw paired endpoint loss", xlabel="Optimizer updates", yscale="log")
axes[1].set(title="Distribution loss (zero for M0/M1)", xlabel="Optimizer updates")
axes[1].set_yscale("symlog", linthresh=1e-7)
for ax in axes[:2]:
    ax.legend()
axes[2].bar(range(len(metrics)), [r["sliced_wasserstein"] for r in metrics])
axes[2].set_xticks(range(len(metrics)), [r["model"] for r in metrics], rotation=45, ha="right")
axes[2].set(title="Quality relative to real validation", ylabel="Fixed-projection SWD ↓")
save_figure(fig, OUTPUT_DIR / "losses_and_quality")
plt.show()


## GIF и интерактивное проигрывание фактических траекторий

Красный — initial noise, фиолетовый — финальный endpoint. Каждая линия хранит историю **той же
частицы**; позиции не масштабируются для анимации. У оригинала N32 и учеников N16 разные
числа Euler-узлов, слайдер показывает реальные индексы и время выбранной истории.

In [ ]:
all_states = torch.cat([result["history"].flatten(0, 1) for result in generations.values()])
common_low, common_high = all_states.min(0).values - .1, all_states.max(0).values + .1
gif_paths = []
if MAKE_GIFS:
    for label, result in generations.items():
        paths = result["history"][:, :GIF_PARTICLES].permute(1, 0, 2)
        path = OUTPUT_DIR / (label.replace(" ", "_") + "_trajectories.gif")
        save_trajectory_gif(paths, path, fps=10, max_frames=65,
                            title=label + " · red noise → purple data",
                            start_label="Start · noise t=1", end_label="Finish · data t=0")
        gif_paths.append(str(path))
    display(Image(filename=gif_paths[0]))
    display(Image(filename=gif_paths[-1]))
print("GIF files:", gif_paths)

if SHOW_WIDGETS:
    selector = widgets.Dropdown(options=list(generations), description="Model")
    frame = widgets.IntSlider(value=0, min=0, max=TEACHER_STEPS, step=1,
                              description="Euler step", continuous_update=False)
    play = widgets.Play(value=0, min=0, max=TEACHER_STEPS, step=1, interval=120)
    link = widgets.jslink((play, "value"), (frame, "value"))
    canvas = widgets.Output()

    def render_frame(change=None):
        result = generations[selector.value]
        last = len(result["history"]) - 1
        frame.max = play.max = last
        index = min(frame.value, last)
        with canvas:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(6, 6))
            state = result["history"][index]
            ax.scatter(state[:, 0], state[:, 1], s=2, alpha=.3, color="gray")
            plot_rainbow_trajectories(result["history"][:, :GIF_PARTICLES].permute(1, 0, 2),
                progress=index / last, ax=ax,
                title=f"{selector.value} · actual Euler index {index}/{last} · t={float(result['times'][index]):.4f}",
                start_label="Start · noise t=1", end_label="Finish · data t=0")
            ax.set_xlim(float(common_low[0]), float(common_high[0]))
            ax.set_ylim(float(common_low[1]), float(common_high[1]))
            plt.show()

    selector.observe(render_frame, names="value")
    frame.observe(render_frame, names="value")
    display(widgets.VBox([widgets.HBox([selector, play, frame]), canvas]))
    render_frame()


## Варианты и полный зарегистрированный Fourier-запуск

`DISTRIBUTION_FAMILY="swd"` меняет distribution loss; `REPRESENTATION="displacement"` или
`"joint"` меняет признаки. `LAMBDA_W=0` должен воспроизвести paired MSE. Изменяйте всю четвёрку
в новом `OUTPUT_DIR`. Поставьте `STUDENT_STEPS=8`, чтобы каждый видимый teacher target делал
четыре микрошага: это отдельная exploratory 32→8 серия с новым каталогом.

Для новой архитектуры замените `student = copy.deepcopy(teacher)` своим `torch.nn.Module`
с `forward(x,t) -> .sample`, уберите только проверку равенства init hash и явно назовите
архитектурную абляцию. Такой student имеет другое начальное условие. Labels не подаются
в модель; сложный conditioner можно добавить своим явным forward и потоком данных.

Teacher обучается обычным Torch FM loop через CLI; это отдельный долгий этап. Все команды
из корня checkout. Их **не нужно** выполнять для повторного exploratory запуска, если
завершённый native teacher уже найден выше. Официальные updates/seeds берутся из YAML,
а не из `UPDATES=200` этой тетрадки:

```bash
python -m recheck.cli verify-losses
python -m recheck.cli train-teacher --config replication/configs/many_rays_fourier_cpu.yaml
# После прерывания этой же teacher-серии добавьте --resume.
python -m recheck.cli teacher-gate --config replication/configs/many_rays_fourier_cpu.yaml
python -m recheck.cli calibrate --config replication/configs/many_rays_fourier_cpu.yaml
python -m recheck.cli distill-suite --config replication/configs/many_rays_fourier_cpu.yaml
```

Для возобновления student-серии добавьте `--resume` к `distill-suite`. Официальная 32→8
проверка использует отдельный `many_rays_fourier_stress_32_8.yaml` и того же замороженного родителя:
optional cell ниже копирует native каталог с manifest, проверяет hash и пишет provenance.
Новый teacher для stress не обучается. После копирования выполните ещё 12 student-прогонов;
**обе серии должны завершиться до первого final/test evaluation**:

```bash
python -m recheck.cli teacher-gate --config replication/configs/many_rays_fourier_stress_32_8.yaml
python -m recheck.cli calibrate --config replication/configs/many_rays_fourier_stress_32_8.yaml
python -m recheck.cli distill-suite --config replication/configs/many_rays_fourier_stress_32_8.yaml
# Все 24 student-прогона завершены; только теперь final/test.
python -m recheck.cli evaluate-suite --config replication/configs/many_rays_fourier_cpu.yaml
python -m recheck.cli evaluate-suite --config replication/configs/many_rays_fourier_stress_32_8.yaml
python -m recheck.cli report --config replication/configs/many_rays_fourier_cpu.yaml \
  --runs replication/artifacts/rays16_fourier_teacher_v1 --output replication/reports/rays16/REPORT.md
python -m recheck.cli report --config replication/configs/many_rays_fourier_stress_32_8.yaml \
  --runs replication/artifacts/rays16_fourier_teacher_stress8_v1 --output replication/reports/rays16_stress8/REPORT.md
```

Для plain comparison явно задайте `TEACHER_PRESET="plain"` и новый `OUTPUT_DIR`.
В CLI у него отдельные `many_rays_cpu.yaml` / `many_rays_stress_32_8.yaml`; их показатели
нельзя выдавать за Fourier-серию. Смена parent требует новой калибровки и свежих students.

В этой тетрадке используется только real validation; final/test CLI-серии остаётся отдельным.


In [ ]:
first_name = next(iter(trained))
restored = load_model(OUTPUT_DIR / first_name / "ema_model", device).eval().requires_grad_(False)
restored_result = sample_euler(restored, initial_noise, steps=STUDENT_STEPS)
assert torch.equal(restored_result["samples"], generations[f"{first_name} N{STUDENT_STEPS}"]["samples"])
assert model_hash(teacher) == teacher_hash
print("Native Diffusers ModelMixin save/load matches; use sample_euler(..., steps=STUDENT_STEPS).")
print("Inspect metrics.json, per-method training.pt / ema_model / manifest.json, PNG/SVG and GIF under", OUTPUT_DIR)


In [ ]:
PREPARE_REGISTERED_STRESS = False     # optional; creates only a shared parent, never trains it
if PREPARE_REGISTERED_STRESS:
    import shutil

    stress_config_path = ("replication/configs/many_rays_fourier_stress_32_8.yaml"
                          if registered_config["model"]["kind"] == "fourier_mlp"
                          else "replication/configs/many_rays_stress_32_8.yaml")
    stress_config = load_config(repo_dir / stress_config_path)
    destination = repo_dir / stress_config["output_dir"] / "teacher"
    if destination.exists():
        raise FileExistsError(f"{destination} already exists; inspect its provenance before reuse")
    if model_hash(teacher) != teacher_hash:
        raise ValueError("Original parent changed; do not prepare a different stress teacher")
    shutil.copytree(teacher_path.parent, destination)
    copied_parent = load_model(destination / "ema_model", "cpu").eval().requires_grad_(False)
    if model_hash(copied_parent) != teacher_hash:
        raise ValueError("Copied parent hash differs; stop before calibration")
    atomic_json(destination.parent / "shared_teacher_provenance.json", {
        "teacher_sha256": teacher_hash, "source": str(teacher_path.parent),
        "source_study": STUDY_ID, "destination_study": stress_config["protocol_id"],
        "completed_teacher_updates": teacher_manifest["updates_completed"],
        "operation": "native directory copy, no retraining or manifest rewriting"})
    print("Copied the exact frozen teacher for registered32→8:", destination)
